# Ten Common pandas Mistakes

Companion to the [complete article](https://todatabeyond.com/blog/maximizing-pandas-efficiency-top-10-mistakes-to-steer-clear-of-in-your-code). Run all cells in order with Python 3.11+ on CPU. No API keys, GPU or data downloads. Pinned pandas 3.1.0, NumPy 2.3.3 and Matplotlib 3.10.9.

All 18 source code examples are retained. sales_data.csv is not supplied by the original: setup explicitly creates a ten-row synthetic fixture, including boundary-date/threshold cases, in a unique temporary directory. It is not the author's original data or a real business dataset. The regression cell cleans up only that directory. Random plot examples use deterministic seeds; current plots are not reconstructions of the original random screenshots.

The deliberately invalid df.Sales Amount expression is compiled and its expected SyntaxError caught without executing a query or arbitrary input. Bracket access and groupby work with spaced column names: the original KeyError claim is wrong. Query expressions are fixed authored strings, never untrusted user input.

Current pandas uses Copy-on-Write; source prose claiming every slice is a mutable view is historical and incorrect. Keep the explicit copy example and verify original data stays unchanged. Method chaining is not fused execution or a universal speed guarantee. Three separate pivots are marginal aggregates, not equivalent to a joint three-key groupby. The final example retains the joint operation and also computes matching separate groupbys; regression checks both semantics independently.

Original storage/dtype/performance prose is retained in the blog and reviewed in the website import review; CSV can carry headers, integer casting is not arbitrary data validation, and Arrow is not itself one universal on-disk format. No Parquet, Feather, HDF5 or Arrow benchmark/serialization execution is claimed. Outputs/counts are empty/null in Git. Actual current outputs and four captured plots are published separately by the website.


In [ ]:
%pip install --quiet --disable-pip-version-check pandas==3.1.0 numpy==2.3.3 matplotlib==3.10.9
from importlib.metadata import version
for package in ['pandas', 'numpy', 'matplotlib']:
    print(package, version(package))


In [ ]:
%matplotlib inline
import warnings
from pathlib import Path
from tempfile import TemporaryDirectory
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

# Explicit synthetic fixture: the source names sales_data.csv but supplies no file.
sales_fixture = pd.DataFrame({
    'order_id': list(range(1, 11)),
    'product_category': ['Electronics'] * 7 + ['Furniture', 'Electronics', 'Electronics'],
    'order_date': ['2022-01-01', '2022-03-01', '2022-02-01', '2022-02-15',
                   '2021-12-31', '2022-03-02', '2022-02-28', '2022-02-01',
                   '2022-01-15', '2022-02-10'],
    'sales_amount': [1000, 100, 999, 1000, 2000, 2000, 1500, 2000, 100, 1000],
    'quantity': [1, 10, 9, 10, 20, 20, 5, 20, 1, 10],
    'region': ['North', 'North', 'South', 'South', 'North', 'South', 'North', 'South', 'North', 'South'],
    'category': ['Hardware', 'Hardware', 'Hardware', 'Hardware', 'Hardware', 'Hardware', 'Accessories', 'Furniture', 'Accessories', 'Accessories'],
    'product': ['Laptop', 'Laptop', 'Laptop', 'Laptop', 'Laptop', 'Laptop', 'Phone', 'Desk', 'Phone', 'Phone'],
    'sales': [1000, 100, 999, 1000, 2000, 2000, 1500, 2000, 100, 1000],
})
workspace = TemporaryDirectory(prefix='pandas-common-mistakes-')
SALES_PATH = Path(workspace.name) / 'sales_data.csv'
sales_fixture.to_csv(SALES_PATH, index=False)
warnings.filterwarnings('error', category=FutureWarning)
warnings.filterwarnings('error', category=pd.errors.ChainedAssignmentError)
print('Synthetic sales fixture:', sales_fixture.shape)
print('CSV written only inside a unique temporary directory.')


## Example 1


In [ ]:
df = pd.DataFrame({'Sales Amount': [100, 100, 250], 'Quantity': [2, 3, 1]})
# The source's intentionally invalid Python is parsed, not executed.
invalid_example = 'df.Sales Amount'
try:
    compile(invalid_example, '<column-name-example>', 'eval')
except SyntaxError as error:
    syntax_error_seen = True
    print(type(error).__name__ + ': ' + error.msg)
else:
    raise AssertionError('Expected the invalid attribute expression to fail')


## Example 2


In [ ]:
space_column = df['Sales Amount']
print(space_column.to_string(index=False))


## Example 3


In [ ]:
space_grouped = df.groupby('Sales Amount')['Quantity'].sum()
print(space_grouped.to_string())


## Example 4


In [ ]:
import pandas as pd

df = pd.DataFrame({'name': ['Alice', 'Bob', 'Charlie', 'David'],
                   'age': [25, 30, 35, 40]})

# Subset where age is greater than 30
subset = df.query('age > 30')
subset_over_30 = subset.copy()
print(subset.to_string(index=False))


## Example 5


In [ ]:
# Subset where age is between 30 and 40, inclusive
subset = df.query('age >= 30 and age <= 40')
subset_inclusive = subset.copy()
print(subset.to_string(index=False))


## Example 6


In [ ]:
import pandas as pd

df = pd.read_csv(SALES_PATH, parse_dates=['order_date'])

product_category = 'Electronics'
start_date = pd.Timestamp('2022-01-01')
end_date = pd.Timestamp('2022-03-01')
min_sales_amount = 1000
min_quantity = 10

subset = df.query('product_category == @product_category and @start_date <= order_date <= @end_date and (sales_amount >= @min_sales_amount or quantity >= @min_quantity)')
sales_subset = subset.copy()
print(subset.to_string(index=False))


## Example 7


In [ ]:
import pandas as pd

df = pd.DataFrame({'x': [1, 2, 3], 'y': [4, 5, 6]})

for index, row in df.iterrows():
    df.loc[index, 'z'] = row['x'] * row['y']
loop_result = df.copy(deep=True)
print(loop_result.to_string(index=False))


## Example 8


In [ ]:
import pandas as pd

df = pd.DataFrame({'x': [1, 2, 3], 'y': [4, 5, 6]})

df['z'] = df['x'] * df['y']
vector_result = df.copy(deep=True)
print(vector_result.to_string(index=False))


## Example 9


In [ ]:
import pandas as pd

# Create a sample DataFrame
data = {'name': ['Alice', 'Bob', 'Charlie', 'David', 'Emily'],
        'age': [25, 32, 18, 47, 29],
        'gender': ['F', 'M', 'M', 'M', 'F']}
df = pd.DataFrame(data)

# Create a copy of the DataFrame
new_df = df.loc[df['age'] > 30, ['name', 'age']].copy()

# Modify the new DataFrame
new_df['age'] = new_df['age'] + 5

# Print both DataFrames
print(df)
print(new_df)
copy_original = df.copy(deep=True)
copy_result = new_df.copy(deep=True)


## Example 10


In [ ]:
import pandas as pd

# Create a sample DataFrame
data = {'name': ['Alice', 'Bob', 'Charlie', 'David', 'Emily'],
        'age': [25, 32, 18, 47, 29],
        'gender': ['F', 'M', 'M', 'M', 'F']}
df = pd.DataFrame(data)

# Apply multiple transformations in a single statement
new_df = (df
          .loc[df['age'] > 30, ['name', 'age']]
          .copy()
          .assign(age_plus_5=lambda x: x['age'] + 5))

# Print the new DataFrame
print(new_df)
chain_original = df.copy(deep=True)
chain_result = new_df.copy(deep=True)


## Example 11


In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

# Create reproducible sample data
rng = np.random.default_rng(52)
df = pd.DataFrame(rng.standard_normal((100, 4)), columns=list('ABCD'))

# Plot a line chart of column A
df['A'].plot(kind='line', color='red', title='Line Plot')
plt.xlabel('Index')
plt.ylabel('Value')
plot_frame_10 = df.copy(deep=True)
print('Shape:', df.shape)
print(df.head().to_string(index=False))
plt.tight_layout()
plt.show()
plt.close('all')


## Example 12


In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

# Create reproducible sample data
rng = np.random.default_rng(53)
df = pd.DataFrame(rng.standard_normal((100, 2)), columns=['X', 'Y'])

# Plot a scatter chart of columns X and Y
df.plot(kind='scatter', x='X', y='Y', color='blue', title='Scatter Plot')
plt.xlabel('X')
plt.ylabel('Y')
plot_frame_11 = df.copy(deep=True)
print('Shape:', df.shape)
print(df.head().to_string(index=False))
plt.tight_layout()
plt.show()
plt.close('all')


## Example 13


In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

# Create reproducible sample data
rng = np.random.default_rng(54)
df = pd.DataFrame({'A': rng.integers(1, 10, 5), 'B': rng.integers(1, 10, 5)})

# Plot a bar chart of columns A and B
df.plot(kind='bar', color=['blue', 'red'], title='Bar Plot')
plt.xlabel('Index')
plt.ylabel('Value')
plot_frame_12 = df.copy(deep=True)
print('Shape:', df.shape)
print(df.head().to_string(index=False))
plt.tight_layout()
plt.show()
plt.close('all')


## Example 14


In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

# Create reproducible sample data
rng = np.random.default_rng(55)
df = pd.DataFrame(rng.standard_normal((1000, 1)), columns=['A'])

# Plot a histogram of column A
df['A'].plot(kind='hist', bins=20, color='green', title='Histogram')
plt.xlabel('Value')
plot_frame_13 = df.copy(deep=True)
print('Shape:', df.shape)
print(df.head().to_string(index=False))
plt.tight_layout()
plt.show()
plt.close('all')


## Example 15


In [ ]:
import pandas as pd

df = pd.read_csv(SALES_PATH)

manual_stats = {}
unique_regions = df['region'].unique()

for region in unique_regions:
    region_sales = df[df['region'] == region]['sales']
    total_sales = region_sales.sum()
    average_sales = region_sales.mean()
    max_sales = region_sales.max()
    min_sales = region_sales.min()
    manual_stats[region] = [total_sales, average_sales, max_sales, min_sales]
    print(f'{region}: Total Sales: {total_sales}, Average Sales: {average_sales}, Max Sales: {max_sales}, Min Sales: {min_sales}')


## Example 16


In [ ]:
import pandas as pd

df = pd.read_csv(SALES_PATH)

region_stats = df.groupby('region')['sales'].agg(['sum', 'mean', 'max', 'min'])
print(region_stats)


## Example 17


In [ ]:
import pandas as pd

df = pd.read_csv(SALES_PATH)

region_sales = pd.pivot_table(df, index='region', values='sales', aggfunc=['sum', 'mean', 'max', 'min'])
category_sales = pd.pivot_table(df, index='category', values='sales', aggfunc=['sum', 'mean', 'max', 'min'])
product_sales = pd.pivot_table(df, index='product', values='sales', aggfunc=['sum', 'mean', 'max', 'min'])

print(region_sales)
print(category_sales)
print(product_sales)


## Example 18


In [ ]:
import pandas as pd

df = pd.read_csv(SALES_PATH)

sales_stats = df.groupby(['region', 'category', 'product'])['sales'].agg(['sum', 'mean', 'max', 'min'])
print(sales_stats)
# Match the separate pivot-table questions with separate groupby operations.
separate_stats = {key: df.groupby(key)['sales'].agg(['sum', 'mean', 'max', 'min'])
                  for key in ['region', 'category', 'product']}
for key, stats in separate_stats.items():
    print(key)
    print(stats.to_string())


## Regression checks


In [ ]:
from pandas.testing import assert_frame_equal, assert_series_equal
assert syntax_error_seen
assert space_column.tolist() == [100, 100, 250]
assert space_grouped.to_dict() == {100: 5, 250: 1}
assert subset_over_30['name'].tolist() == ['Charlie', 'David']
assert subset_inclusive['age'].tolist() == [30, 35, 40]
assert sales_subset['order_id'].tolist() == [1, 2, 4, 7, 10]
dates = pd.to_datetime(sales_fixture['order_date'])
expected = sales_fixture.loc[(sales_fixture['product_category'] == 'Electronics') &
                            dates.between(start_date, end_date) &
                            ((sales_fixture['sales_amount'] >= 1000) | (sales_fixture['quantity'] >= 10))].copy()
expected['order_date'] = pd.to_datetime(expected['order_date'])
assert_frame_equal(sales_subset, expected)
assert vector_result['z'].tolist() == loop_result['z'].tolist() == [4, 10, 18]
np.testing.assert_array_equal(loop_result.to_numpy(), vector_result.to_numpy())
assert copy_original['age'].tolist() == [25, 32, 18, 47, 29]
assert copy_result['age'].tolist() == [37, 52]
assert chain_result['age_plus_5'].tolist() == [37, 52]
assert chain_result['age'].tolist() == [32, 47]
assert_frame_equal(copy_original, chain_original)
# pandas 3 Copy-on-Write also keeps an ordinary slice independent on assignment.
cow_slice = chain_original[['age']]
cow_slice['age'] = 0
assert chain_original['age'].tolist() == [25, 32, 18, 47, 29]
for i, shape in [(10, (100, 4)), (11, (100, 2)), (12, (5, 2)), (13, (1000, 1))]:
    assert globals()['plot_frame_' + str(i)].shape == shape
assert np.isfinite(plot_frame_13.to_numpy()).all()
assert plot_frame_12.ge(1).all().all() and plot_frame_12.lt(10).all().all()
manual_frame = pd.DataFrame.from_dict(manual_stats, orient='index', columns=['sum', 'mean', 'max', 'min'])
manual_frame.index.name = 'region'
assert_frame_equal(region_stats, manual_frame.sort_index(), check_dtype=False)
for key, pivot in [('region', region_sales), ('category', category_sales), ('product', product_sales)]:
    flattened = pivot.copy()
    flattened.columns = flattened.columns.get_level_values(0)
    assert_frame_equal(separate_stats[key], flattened, check_names=False)
assert_frame_equal(sales_stats, sales_fixture.groupby(['region', 'category', 'product'])['sales'].agg(['sum', 'mean', 'max', 'min']))
assert sales_stats['sum'].sum() == region_stats['sum'].sum() == sales_fixture['sales'].sum()
assert len(sales_stats) > len(region_stats)
assert_frame_equal(pd.read_csv(SALES_PATH), sales_fixture)
assert len(list(Path(workspace.name).iterdir())) == 1
workspace.cleanup()
print('All column-access, query boundary, vectorization, copy, chaining, plot-data and aggregation checks passed.')
print('Sales fixture: 10 synthetic rows; complex query matched order IDs [1, 2, 4, 7, 10].')
print('Four plot datasets checked; separate marginal and joint aggregation semantics verified.')
